In [ ]:
# data download
import pandas as pd

base_url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/"

files = {
    "demo": "DEMO_J.xpt",
    "ohxden": "OHXDEN_J.xpt",
    "ohq": "OHQ_J.xpt",
    "ohxref": "OHXREF_J.xpt"
}

dfs = {}
for name, file in files.items():
    url = base_url + file
    print(f"Loading {file}...")
    dfs[name] = pd.read_sas(url, format="xport", encoding="utf-8")
    print(f"  -> {dfs[name].shape[0]} rows, {dfs[name].shape[1]} cols")

# Merge all on SEQN — left join from DEMO as the anchor
df = dfs["demo"]
for name in ["ohxden", "ohq", "ohxref"]:
    df = df.merge(dfs[name], on="SEQN", how="left")

print(f"\nMerged shape: {df.shape}")


Loading DEMO_J.xpt...
  -> 9254 rows, 46 cols
Loading OHXDEN_J.xpt...
  -> 8366 rows, 170 cols
Loading OHQ_J.xpt...
  -> 8897 rows, 44 cols
Loading OHXREF_J.xpt...
  -> 8366 rows, 12 cols

Merged shape: (9254, 269)


In [2]:
# Keep only participants who completed both interview AND MEC exam
# RIDSTATR == 2 means interviewed + examined
df = df[df["RIDSTATR"] == 2].copy()
print(f"After filtering to examined participants: {df.shape}")

# Sanity check — should be close to 8,366
# (small differences possible due to exam completion edge cases)

# Drop columns that are purely administrative/not analytically useful
admin_cols = [
    "SDDSRVYR",   # data release cycle (constant)
    "RIDEXMON",   # 6-month exam period
    "RIDAGEMN",   # age in months (redundant with RIDAGEYR for adults)
    "RIDEXAGM",   # age in months at exam (only populated for under 19)
    "SIALANG", "SIAPROXY", "SIAINTRP",  # interview language/proxy flags
    "FIALANG", "FIAPROXY", "FIAINTRP",
    "MIALANG", "MIAPROXY", "MIAINTRP",
    "AIALANGA",
]
df = df.drop(columns=[c for c in admin_cols if c in df.columns])
print(f"After dropping admin columns: {df.shape}")

# Quick missingness summary on key variables
key_vars = ["INDFMPIR", "RIDAGEYR", "RIAGENDR", "RIDRETH3", 
            "DMDEDUC2", "INDHHIN2", "OHQ845", "OHAREC"]
print("\nMissingness in key variables:")
print(df[key_vars].isnull().sum())

After filtering to examined participants: (8704, 269)
After dropping admin columns: (8704, 255)

Missingness in key variables:
INDFMPIR    1070
RIDAGEYR       0
RIAGENDR       0
RIDRETH3       0
DMDEDUC2    3439
INDHHIN2     384
OHQ845       340
OHAREC       604
dtype: int64


In [3]:
# Check age distribution of DMDEDUC2 missingness
# (confirm it's just under-20s)
print("Age breakdown where DMDEDUC2 is missing:")
print(df[df["DMDEDUC2"].isnull()]["RIDAGEYR"].describe())

# How many are adults (20+) with missing education?
adults_missing_educ = df[(df["RIDAGEYR"] >= 20) & (df["DMDEDUC2"].isnull())]
print(f"\nAdults 20+ missing DMDEDUC2: {len(adults_missing_educ)}")

# INDFMPIR missingness — is it random or skewed toward certain groups?
print("\nMean age where INDFMPIR is missing vs present:")
print(df.groupby(df["INDFMPIR"].isnull())["RIDAGEYR"].mean())
print("\nRace breakdown where INDFMPIR is missing:")
print(df[df["INDFMPIR"].isnull()]["RIDRETH3"].value_counts())

Age breakdown where DMDEDUC2 is missing:
count    3.439000e+03
mean     8.454783e+00
std      5.893142e+00
min      5.397605e-79
25%      3.000000e+00
50%      8.000000e+00
75%      1.300000e+01
max      1.900000e+01
Name: RIDAGEYR, dtype: float64

Adults 20+ missing DMDEDUC2: 0

Mean age where INDFMPIR is missing vs present:
INDFMPIR
False    34.133482
True     36.615888
Name: RIDAGEYR, dtype: float64

Race breakdown where INDFMPIR is missing:
RIDRETH3
4.0    303
1.0    206
3.0    186
2.0    159
6.0    158
7.0     58
Name: count, dtype: int64


In [4]:
# Restrict to adults 20+
df_adults = df[df["RIDAGEYR"] >= 20].copy()
print(f"Adult sample (20+): {df_adults.shape}")

# Re-check missingness on key vars in adult sample
print("\nMissingness in key variables (adults only):")
print(df_adults[key_vars].isnull().sum())

# Check INDFMPIR distribution before imputation
print("\nINDFMPIR summary:")
print(df_adults["INDFMPIR"].describe())

Adult sample (20+): (5265, 255)

Missingness in key variables (adults only):
INDFMPIR    697
RIDAGEYR      0
RIAGENDR      0
RIDRETH3      0
DMDEDUC2      0
INDHHIN2    263
OHQ845        0
OHAREC      208
dtype: int64

INDFMPIR summary:
count    4.568000e+03
mean     2.553380e+00
std      1.603734e+00
min      5.397605e-79
25%      1.200000e+00
50%      2.130000e+00
75%      4.080000e+00
max      5.000000e+00
Name: INDFMPIR, dtype: float64


In [5]:
# Fix the near-zero minimum — treat as true 0 or near-poverty
df_adults["INDFMPIR"] = df_adults["INDFMPIR"].clip(lower=0)

# Impute INDFMPIR missing values with median
indfmpir_median = df_adults["INDFMPIR"].median()
df_adults["INDFMPIR"] = df_adults["INDFMPIR"].fillna(indfmpir_median)
print(f"INDFMPIR median used for imputation: {indfmpir_median:.3f}")

# For INDHHIN2 and OHAREC, check what's missing
print(f"\nOHAREC value counts (including NaN):")
print(df_adults["OHAREC"].value_counts(dropna=False))

print(f"\nINDHHIN2 missing: {df_adults['INDHHIN2'].isnull().sum()} — will drop later if not used as primary predictor")

# Final missingness check
print(f"\nRemaining missingness in key vars:")
print(df_adults[key_vars].isnull().sum())

INDFMPIR median used for imputation: 2.130

OHAREC value counts (including NaN):
OHAREC
4.0    2910
3.0    1792
2.0     348
NaN     208
1.0       7
Name: count, dtype: int64

INDHHIN2 missing: 263 — will drop later if not used as primary predictor

Remaining missingness in key vars:
INDFMPIR      0
RIDAGEYR      0
RIAGENDR      0
RIDRETH3      0
DMDEDUC2      0
INDHHIN2    263
OHQ845        0
OHAREC      208
dtype: int64


In [ ]:
import numpy as np

# ── CTC LETTER CODE MAPPING (NHANES OHXDEN codebook) ──
# S = Sound/healthy (no caries, no restoration)
# F = Filled, no caries (treated, sound)
# E = Missing due to dental disease/caries
# P = Decayed (untreated caries, no restoration)
# Z = Tooth with sealant only (sound)
# J = Filled with caries (restoration + active decay)
# Q = Filled with caries (alternate code, same as J)
# M = Missing due to other reason (trauma, ortho, congenital)
# R = Root tip/fragment only remaining
# NaN / blank = tooth not present / not assessed

# ── TC NUMERIC CODE MAPPING ──
# 2 = Permanent tooth present
# 3 = Dental implant
# 4 = Tooth not present (missing/extracted)
# 5 = Permanent tooth present (alternate code — included alongside 2)
# 9 = Cannot assess

# ── TOOTH COUNT FEATURES (TC columns, numeric) ──
tc_cols = [c for c in df_adults.columns
           if c.startswith("OHX") and c.endswith("TC") and "CTC" not in c]

df_adults["teeth_present"] = df_adults[tc_cols].isin([2, 5]).sum(axis=1)  # codes 2 & 5 both = permanent present
df_adults["teeth_missing"] = (df_adults[tc_cols] == 4).sum(axis=1)
df_adults["teeth_implant"] = (df_adults[tc_cols] == 3).sum(axis=1)

# ── CORONAL CARIES FEATURES (CTC columns, letter codes) ──
ctc_cols = [c for c in df_adults.columns
            if c.startswith("OHX") and c.endswith("CTC")]

df_adults["teeth_decayed"]        = df_adults[ctc_cols].isin(["P"]).sum(axis=1)         # active untreated decay
df_adults["teeth_filled_decayed"] = df_adults[ctc_cols].isin(["J", "Q"]).sum(axis=1)   # filled but still decayed
df_adults["teeth_filled_sound"]   = df_adults[ctc_cols].isin(["F"]).sum(axis=1)         # filled, no decay
df_adults["teeth_missing_caries"] = df_adults[ctc_cols].isin(["E"]).sum(axis=1)         # missing due to caries
df_adults["teeth_sound"]          = df_adults[ctc_cols].isin(["S", "Z"]).sum(axis=1)    # sound / sealant only
df_adults["teeth_root_tip"]       = df_adults[ctc_cols].isin(["R"]).sum(axis=1)         # root tip/fragment

# ── DERIVED SUMMARY FEATURES ──
# Classic DMFT: Decayed + Missing (caries) + Filled
df_adults["dmft_score"] = (
    df_adults["teeth_decayed"] +
    df_adults["teeth_filled_decayed"] +
    df_adults["teeth_missing_caries"] +
    df_adults["teeth_filled_sound"]
)

# Total active decay burden (untreated)
df_adults["total_decay_burden"] = (
    df_adults["teeth_decayed"] + df_adults["teeth_filled_decayed"]
)

# Untreated decay ratio (out of all assessed teeth)
total_assessed = df_adults[ctc_cols].notna().sum(axis=1)
df_adults["untreated_decay_ratio"] = (
    df_adults["total_decay_burden"] / total_assessed.replace(0, np.nan)
)

# Treatment ratio: filled / (filled + decayed) — access to care proxy
ever_decayed = (df_adults["teeth_decayed"] +
                df_adults["teeth_filled_decayed"] +
                df_adults["teeth_filled_sound"])
df_adults["treatment_ratio"] = (
    df_adults["teeth_filled_sound"] / ever_decayed.replace(0, np.nan)
)

# ── SUMMARY ──
new_features = ["teeth_present", "teeth_missing", "teeth_implant",
                "teeth_sound", "teeth_decayed", "teeth_filled_decayed",
                "teeth_filled_sound", "teeth_missing_caries", "teeth_root_tip",
                "dmft_score", "total_decay_burden",
                "untreated_decay_ratio", "treatment_ratio"]

print("Engineered feature summary:")
print(df_adults[new_features].describe().round(3))

print(f"\nParticipants with any active decay: {(df_adults['teeth_decayed'] > 0).sum()}")
print(f"Mean DMFT score: {df_adults['dmft_score'].mean():.2f}")

In [ ]:
# ── TARGET 1: Binary classification — Poor oral health (self-reported) ──
# OHQ845: 1=Excellent, 2=Very good, 3=Good, 4=Fair, 5=Poor
# Binarize: 1 = Fair or Poor (4-5), 0 = Excellent/Very good/Good (1-3)
df_adults["target_poor_selfrated"] = (df_adults["OHQ845"] >= 4).astype(int)
print("Target 1 — Poor self-rated oral health:")
print(df_adults["target_poor_selfrated"].value_counts())
print(f"Positive rate: {df_adults['target_poor_selfrated'].mean():.3f}")

# ── TARGET 2: Binary classification — Needs dental care (clinical) ──
# OHAREC: 1=Immediate, 2=Soon, 3=When time permits, 4=No obvious problem
# Binarize: 1 = Any referral (1-3), 0 = No obvious problem (4)
# .where() preserves NaN for the 208 missing OHAREC values instead of silently coding them as 0
df_adults["target_needs_care"] = (
    (df_adults["OHAREC"] < 4)
    .where(df_adults["OHAREC"].notna())
    .astype("Int64")
)
print("\nTarget 2 — Clinically needs dental care:")
print(df_adults["target_needs_care"].value_counts(dropna=False))
print(f"Positive rate (non-missing): {df_adults['target_needs_care'].mean():.3f}")
print(f"Missing (excluded from target): {df_adults['target_needs_care'].isna().sum()}")

# ── REGRESSION TARGET: Continuous — DMFT score ──
print(f"\nRegression target — DMFT score:")
print(df_adults["dmft_score"].describe().round(2))

In [ ]:
# ── FINAL EXPORT ──
import os

output_dir = os.path.join(os.path.dirname(os.path.abspath("data_explore.ipynb")), "data")
os.makedirs(output_dir, exist_ok=True)

# Parquet preserves dtypes (including nullable Int64 for target_needs_care)
parquet_path = os.path.join(output_dir, "nhanes_oral_health_adults.parquet")
df_adults.to_parquet(parquet_path, index=False)
print(f"Saved: {parquet_path}")

# CSV for easy inspection / sharing
csv_path = os.path.join(output_dir, "nhanes_oral_health_adults.csv")
df_adults.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

print(f"\nFinal dataset: {df_adults.shape[0]:,} rows × {df_adults.shape[1]} columns")

# Confirm all engineered features and targets are present
expected = [
    "teeth_present", "teeth_missing", "teeth_implant", "teeth_sound",
    "teeth_decayed", "teeth_filled_decayed", "teeth_filled_sound",
    "teeth_missing_caries", "teeth_root_tip",
    "dmft_score", "total_decay_burden", "untreated_decay_ratio", "treatment_ratio",
    "target_poor_selfrated", "target_needs_care"
]
missing_cols = [c for c in expected if c not in df_adults.columns]
if missing_cols:
    print(f"\n⚠️  Missing expected columns: {missing_cols}")
else:
    print(f"\n✓ All {len(expected)} engineered features and targets present")